In [25]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt

REPLACE = False


In [26]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [27]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")
embeddings_db_path = os.path.join(rel_path, "embeddings.db")
batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)
embedding_db_conn = llt.get_embedding_store_db_conn(embeddings_db_path)

In [28]:
batch_jobs = llt.get_batch_jobs_df(batch_db_conn)
for _, row in batch_jobs.iterrows():
    batch_id = row['id']
    status = row['status']
    if status not in ['completed', 'failed', 'cancelled', 'written', 'error']:
        batch = llt.check_batch_status(batch_id, rel_path, batch_db_conn)
        print(f"Batch {batch_id} status updated to {batch.status}")

Batch msgbatch_0166jrEzkrDBzW6q8TPbEgrq status updated to completed


In [29]:
batch_jobs = llt.get_batch_jobs_df(batch_db_conn)
batch_jobs = batch_jobs.loc[batch_jobs['status'] != 'written']

for _, row in batch_jobs.iterrows():
    batch_id = row['id']
    status = row['status']
    input_filename = row['input_file']
    output_filename = row['output_file']
    type = row['type']
    if (status == 'completed') or (status == 'expired'):
        print(f"Batch {batch_id}: ", end="", flush=True)
        if type=='chat':
            result = llt.write_batch_to_response_store(batch_id, rel_path, batch_db_conn, response_db_conn, overwrite=REPLACE)
        elif type=='embedding':
            result = llt.write_batch_to_embedding_store(batch_id, rel_path, batch_db_conn, embedding_db_conn, overwrite=REPLACE)
        if result is not None:
            llt.update_batch_job(batch_id, status='written', db_conn=batch_db_conn)
            input_filepath = os.path.join(DATA_PATH, rel_path, input_filename)
            output_filepath = os.path.join(DATA_PATH, rel_path, output_filename)
            if os.path.exists(input_filepath) and len(input_filename)>0:
                os.remove(input_filepath)
            if os.path.exists(output_filepath) and len(output_filename)>0:
                os.remove(output_filepath)
        else:
            llt.update_batch_job(batch_id, status='error', db_conn=batch_db_conn)

Batch msgbatch_0166jrEzkrDBzW6q8TPbEgrq: Chat batch job msgbatch_0166jrEzkrDBzW6q8TPbEgrq - Total requests: 330, Written: 330, Existing: 0, Errors: 0


In [30]:
batch_db_conn.close()
response_db_conn.close()
embedding_db_conn.close()